In [6]:
import pandas as pd
import os

dataset_path = "../dataset/raw/iam_words"

labels = []

with open(f"{dataset_path}/words.txt", "r") as f:
    for line in f:
        if line.startswith("#"):
            continue

        parts = line.strip().split()

        image_id = parts[0]
        word = parts[-1]

        folder1 = image_id.split("-")[0]
        folder2 = "-".join(image_id.split("-")[:2])

        image_path = f"{dataset_path}/words/{folder1}/{folder2}/{image_id}.png"

        if os.path.exists(image_path):
            labels.append([image_path, word])

df = pd.DataFrame(labels, columns=["image_path", "label"])

print("Total samples:", len(df))
df.head()

Total samples: 44564


,image_path,label
0,../dataset/raw/iam_words/words/a01/a01-000u/a0...,A
1,../dataset/raw/iam_words/words/a01/a01-000u/a0...,MOVE
2,../dataset/raw/iam_words/words/a01/a01-000u/a0...,to
3,../dataset/raw/iam_words/words/a01/a01-000u/a0...,stop
4,../dataset/raw/iam_words/words/a01/a01-000u/a0...,Mr.


In [7]:
df.to_csv("../dataset/processed/labels.csv", index=False)
print("labels.csv created ✅")

labels.csv created ✅


In [8]:
# remove empty labels
df = df[df['label'].notnull()]

# remove labels with non-alphabet (optional but recommended)
df = df[df['label'].str.isalpha()]

print("After cleaning:", len(df))

After cleaning: 37827


In [9]:
from sklearn.model_selection import train_test_split

# first split: train + temp
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

# second split: val + test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

Train size: 30261
Val size: 3783
Test size: 3783


In [10]:
train_df.to_csv("../dataset/processed/train.csv", index=False)
val_df.to_csv("../dataset/processed/val.csv", index=False)
test_df.to_csv("../dataset/processed/test.csv", index=False)

print("train.csv, val.csv, test.csv created ✅")

train.csv, val.csv, test.csv created ✅


In [12]:
import pandas as pd
import cv2

df = pd.read_csv("../dataset/processed/train.csv")

valid_rows = []

for _, row in df.iterrows():
    path = row["image_path"]

    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    # ✅ STRICT check (not just path existence)
    if img is not None:
        valid_rows.append(row)

df_clean = pd.DataFrame(valid_rows)

df_clean.to_csv("../dataset/processed/train_clean.csv", index=False)

print("Original:", len(df))
print("Cleaned :", len(df_clean))

Original: 30261
Cleaned : 30260


In [13]:
# =========================
# 1️⃣ IMPORTS
# =========================
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
from torch.utils.data import DataLoader

from src.dataset import IAMDataset
from src.model import CRNN
from src.train import train


# =========================
# 2️⃣ LOAD DATA
# =========================
train_df = pd.read_csv("../dataset/processed/train_clean.csv")

# =========================
# 2️⃣ VALIDATE DATASET 🔍 (PUT HERE)
# =========================
import cv2

for path in train_df['image_path']:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print("Bad:", path)
        break
else:
    print("All images valid ✅")




# =========================
# 3️⃣ CHARACTER MAPPING ✅
# =========================
chars = sorted(list(set("".join(train_df['label']))))

char_to_idx = {c: i+1 for i, c in enumerate(chars)}
idx_to_char = {i+1: c for i, c in enumerate(chars)}

num_classes = len(chars) + 1  # +1 for CTC blank


# =========================
# 4️⃣ COLLATE FUNCTION ✅ (MUST BE BEFORE DATALOADER)
# =========================
def collate_fn(batch):
    images, labels = zip(*batch)
    images = torch.stack(images, 0)
    return images, labels


# =========================
# 5️⃣ DATASET + DATALOADER
# =========================
train_dataset = IAMDataset(train_df, char_to_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)


# =========================
# 6️⃣ MODEL
# =========================
model = CRNN(num_classes)


# =========================
# 7️⃣ LOSS + OPTIMIZER
# =========================
criterion = nn.CTCLoss(blank=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

import importlib
import src.train

importlib.reload(src.train)

from src.train import train
# =========================
# 1️⃣ Training done above
# =========================

for epoch in range(20):
    print(f"Epoch {epoch+1}")
    train(model, train_loader, optimizer, criterion)


# =========================
# 2️⃣ ADD decode function HERE ✅
# =========================
def decode_prediction(preds, idx_to_char):
    preds = preds.argmax(2)
    preds = preds.permute(1, 0)

    texts = []

    for pred in preds:
        prev = -1
        text = ""
        for p in pred:
            p = p.item()
            if p != prev and p != 0:
                text += idx_to_char.get(p, "")
            prev = p
        texts.append(text)

    return texts


# =========================
# 3️⃣ TEST PREDICTION
# =========================
model.eval()

for imgs, labels in train_loader:
    preds = model(imgs)

    texts = decode_prediction(preds, idx_to_char)

    print("Predicted:", texts[0])
    print("Actual   :", "".join([idx_to_char[i.item()] for i in labels[0]]))
    break

All images valid ✅
Epoch 1
Loss: 2.9022306140835137
Epoch 2
Loss: 1.677700478544195
Epoch 3
Loss: 1.071389814007862
Epoch 4
Loss: 0.7373905281750899
Epoch 5
Loss: 0.5577613014814465
Epoch 6
Loss: 0.439044715202636
Epoch 7
Loss: 0.34218805410109665
Epoch 8
Loss: 0.2718149100924521
Epoch 9
Loss: 0.21626377328994031
Epoch 10
Loss: 0.17284155619698902
Epoch 11
Loss: 0.1358205879158311
Epoch 12
Loss: 0.1230262263183803
Epoch 13
Loss: 0.10538366078999657
Epoch 14
Loss: 0.08053923245562716
Epoch 15
Loss: 0.08219975692832382
Epoch 16
Loss: 0.06663666432378833
Epoch 17
Loss: 0.07113560604179636
Epoch 18
Loss: 0.060677047496480396
Epoch 19
Loss: 0.04946745800255083
Epoch 20
Loss: 0.05075122484195009
Predicted: tatFwlaMacwotWtRafStctfaIApw
Actual   : the
